# GReaT colab example

**Please make sure you select a runtime with GPU**

*(Runtime-> Change Runtime-> Hardware accelerator is set to GPU)*

In [1]:
!pip install --upgrade be-great sentencepiece peft -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.1/519.1 kB 20.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 58.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 60.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 27.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 31.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 108.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver doe

In [2]:
# !huggingface-cli login

In [40]:
from be_great import GReaT
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True).frame

model_name = 'gpt2' # @param ["gpt2", "distilgpt2", "rinna/japanese-gpt2-medium", "meta-llama/Llama-2-7b-hf"] {allow-input: true}

model = GReaT(
    llm=model_name,
    batch_size=64,
    epochs=1,
    save_steps=100,
    # efficient_finetuning='lora'
    )

model.fit(data)

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss


In [44]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [45]:
start_tokens = tokenizer("MedInc is ", return_tensors="pt").to("cuda:0")
start_tokens

{'input_ids': tensor([[ 9921, 25517,   318,   220]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1]], device='cuda:0')}

In [47]:
import torch

with torch.no_grad():
    outputs = model.model.generate(
        **start_tokens,
        max_length=100,
        do_sample=True,
        top_k=10,
        temperature=.7,
        pad_token_id=tokenizer.pad_token_id,
    )

text_data = [tokenizer.decode(t) for t in outputs]
text_data = [d.replace("<|endoftext|>", "") for d in text_data]
text_data = [d.replace("\n", " ") for d in text_data]
text_data = [d.replace("\r", "") for d in text_data]

text_data

['MedInc is 23.0, AveOccup is 3.83333333333333333, Latitude is 34.02, Longitude is -118.28, Population is 561.0, AveBedrms is 1.09090909090909, HouseAge is 52.0, AveRooms is 4.0, MedHouseVal is 0.857']

In [48]:
data.columns

Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude', 'MedHouseVal'],
      dtype='object')

In [49]:
synthetic_data = model.sample(n_samples=1_000)

1032it [00:19, 52.68it/s]


In [50]:
synthetic_data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,5.382068,23.0,5.062119,0.918207,1132.0,2.938207,36.31,-118.28,5.00001
1,3.545500,27.0,5.350000,1.074000,946.0,2.967000,34.08,-122.02,1.06000
2,2.518800,23.0,5.167747,1.167747,7.0,2.908200,34.03,-118.06,1.99700
3,3.010000,45.0,5.788980,1.000000,627.0,2.856193,33.96,-118.26,1.35200
4,2.456700,16.0,4.603925,1.053589,1059.0,2.908255,34.80,-121.15,1.87500
